# Eval-Driven Development

In this notebook we'll walk through **Eval-Driven Development (EDD)** — an approach to building AI agents inspired by test-driven development.

The central idea: **define how you'll measure success before (or alongside) building the application.** This forces you to think clearly about what "good" looks like, and lets you iterate confidently.

We'll use the email agent from the first session as our application and build a full evaluation suite around it.

---

### The 3 Components of Offline Evaluation
![flow](../images/evals-conceptual.png)

**Critically, you can define components 1 and 3 _before_ you have an application.**

## Setup

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env")

from langsmith import Client
from langchain_openai import ChatOpenAI

client = Client()
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
# Ensure prompts and datasets are available in LangSmith before we start
from utils.setup_langsmith import main
main()

---
## Component 1: Dataset

A dataset is a collection of **labeled examples** representing the inputs your agent will encounter and the expected outputs.

### Starting point: manual curation

Before your agent is in production, you won't have real user data. That's fine — start by manually writing a handful of representative examples.

The key question: **what are the cases your agent needs to handle well?**

For our email agent, we care about:
1. **Triage accuracy** — does it correctly classify emails as `ignore`, `notify`, or `respond`?
2. **Response quality** — when it responds, does it hit the right criteria?
3. **Tool usage** — does it call the right tools in the right order?

In [ ]:
# A few manually curated examples to illustrate the idea
# Each example has: the input + the expected output (label)

example_1 = {
    "input": {
        "email_input": {
            "author": "Alice Smith <alice.smith@company.com>",
            "to": "Robert Xu <Robert@company.com>",
            "subject": "Quick question about API documentation",
            "email_thread": """Hi Robert,

I was reviewing the API documentation for the new authentication service and noticed a few endpoints seem to be missing from the specs. Could you help clarify if this was intentional or if we should update the docs?

Specifically, I'm looking at:
- /auth/refresh
- /auth/validate

Thanks!
Alice"""
        }
    },
    "output": {
        "classification": "respond",
        "response_criteria": "Send email with write_email tool to acknowledge the question and confirm it will be investigated",
        "trajectory": ["write_email", "Done"]
    }
}

example_2 = {
    "input": {
        "email_input": {
            "author": "Marketing Team <marketing@company.com>",
            "to": "Robert Xu <Robert@company.com>",
            "subject": "New Company Newsletter Available",
            "email_thread": "Hello Robert,\n\nThe latest edition of our company newsletter is now available on the intranet.\n\nBest regards,\nMarketing Team"
        }
    },
    "output": {
        "classification": "ignore",
        "response_criteria": "No response needed. Ensure this is ignored.",
        "trajectory": []
    }
}

example_3 = {
    "input": {
        "email_input": {
            "author": "System Admin <sysadmin@company.com>",
            "to": "Robert Xu <Robert@company.com>",
            "subject": "Scheduled maintenance - database downtime",
            "email_thread": "Hi Robert,\n\nThis is a reminder that we'll be performing scheduled maintenance on the production database tonight from 2AM to 4AM EST. During this time, all database services will be unavailable.\n\nPlease plan your work accordingly.\n\nThanks,\nSystem Admin Team"
        }
    },
    "output": {
        "classification": "notify",
        "response_criteria": "No response needed. Ensure the user is notified.",
        "trajectory": []
    }
}

In [ ]:
# Push these examples to LangSmith as a dataset
# This gives us a versioned, reusable set of test cases

mini_dataset_name = "Email Agent: Mini Demo"

if not client.has_dataset(dataset_name=mini_dataset_name):
    examples = [example_1, example_2, example_3]
    dataset = client.create_dataset(mini_dataset_name)
    client.create_examples(
        inputs=[e["input"] for e in examples],
        outputs=[e["output"] for e in examples],
        dataset_id=dataset.id,
    )
    print(f"Created dataset: {mini_dataset_name}")
else:
    print(f"Dataset already exists: {mini_dataset_name}")

### Loading the full dataset

In practice, you'll grow this dataset over time — adding real production examples, synthetic data, and edge cases you discover.

We've already built a dataset of **16 labeled emails** across all three categories. Let's load that into LangSmith.

In [ ]:
from utils.datasets import load_triage_datasets, load_response_datasets, load_trajectory_datasets

triage_dataset = load_triage_datasets()
response_dataset = load_response_datasets()
trajectory_dataset = load_trajectory_datasets()

print(f"\nDatasets ready:")
print(f"  - {triage_dataset}")
print(f"  - {response_dataset}")
print(f"  - {trajectory_dataset}")

---
## Component 2: The Run Function

The run function wraps your agent so that LangSmith can call it on each dataset example and record the results.

### How do you define it before you have an application?

> *"Humans are very effective agents. How a human tackles a problem will often be an effective starting point for how agents will tackle that problem."*

Think through how **you** would handle incoming email:

![flow](../images/email_agent.png)

This mental model becomes your **run function** — the agent mimics the human workflow.

We already built this agent in the first session. Here we import it and wrap it for evaluation.

In [ ]:
from utils.agent import email_assistant

def run_email_agent(inputs: dict) -> dict:
    result = email_assistant.invoke({"email_input": inputs["email_input"]})

    tool_calls_made = []
    written_emails = []

    for msg in result.get("messages", []):
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                tool_calls_made.append(tc["name"])
                if tc["name"] == "write_email":
                    written_emails.append(tc["args"])

    return {
        "classification_decision": result.get("classification_decision", "unknown"),
        "tool_calls": tool_calls_made,
        "written_emails": written_emails,
    }

In [ ]:
from utils.datasets import email_input_1

test_output = run_email_agent({"email_input": email_input_1})

print("Classification:", test_output["classification_decision"])
print("Tools called:", test_output["tool_calls"])
if test_output["written_emails"]:
    print("\nEmail written:")
    for email in test_output["written_emails"]:
        print(f"  To: {email.get('to')}")
        print(f"  Subject: {email.get('subject')}")
        print(f"  Content: {email.get('content', '')[:200]}...")

---
## Component 3: Evaluators

Evaluators measure whether your agent's output is good. There are two flavors:

| Type | When to use | Example |
|---|---|---|
| **Code-based** | Structured output with a deterministic expected value | Triage classification label, tool call list |
| **LLM-as-judge** | Open-ended output where "correctness" requires judgment | Quality of a written email |

Evaluator functions in LangSmith have this signature:
```python
def my_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    # inputs: the dataset example's input
    # outputs: what your run function returned
    # reference_outputs: the labeled ground truth
    return {"key": "metric_name", "score": 0 or 1}
```

### Code Evaluators: Structured Output

#### Eval 1: Triage Classification

The triage decision (`ignore / notify / respond`) is a discrete label — a simple exact-match is the right evaluator here.

In [ ]:
def correct_label(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    predicted = outputs.get("classification_decision", "").lower()
    expected = reference_outputs.get("classification", "").lower()
    return {"key": "correct_label", "score": int(predicted == expected)}

In [ ]:
from langsmith import evaluate

triage_results = evaluate(
    run_email_agent,
    data=triage_dataset,
    evaluators=[correct_label],
    experiment_prefix="email-triage-v1",
)

# View results in LangSmith, or inspect inline
triage_results.to_pandas()[["inputs.email_input", "outputs.classification_decision", "feedback.correct_label"]].head()

#### Eval 2: Tool Call Trajectory

Beyond the final decision, we want to check: did the agent use the **right tools**?

For example, a meeting request should trigger `check_calendar_availability` before `write_email`. We can check whether the set of tools called matches what we'd expect.

In [ ]:
def trajectory_match(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    expected = set(t.lower() for t in reference_outputs.get("trajectory", []))
    actual = set(t.lower() for t in outputs.get("tool_calls", []))
    return {"key": "trajectory_match", "score": int(expected == actual)}

In [ ]:
trajectory_results = evaluate(
    run_email_agent,
    data=trajectory_dataset,
    evaluators=[trajectory_match],
    experiment_prefix="email-trajectory-v1",
)

trajectory_results.to_pandas()[["outputs.tool_calls", "feedback.trajectory_match"]].head(10)

### LLM-as-Judge: Unstructured Output

#### Eval 3: Final Response Quality

For the actual email the agent writes, there's no single correct answer — many phrasings could be valid. We need an LLM to judge whether the response meets the criteria.

Our rubric for each example specifies exactly what the response should accomplish:
- Did it acknowledge the deadline?
- Did it check calendar availability before scheduling?
- Did it ask the right follow-up questions?

This is where **LLM-as-judge** shines: it can evaluate nuanced, open-ended quality the same way a human reviewer would.

In [ ]:
from pydantic import BaseModel, Field

class Completeness(BaseModel):
    completeness: bool = Field(
        description="Does the output generated by the agent meet all the success criteria defined in the reference output?"
    )

judge = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(Completeness)

COMPLETENESS_SYSTEM = """
You are an expert data analyst grading outputs generated by an AI email assistant.
You are to judge whether the agent generated an accurate and complete response for the given input email.
You are also provided with success criteria written by a human, which serves as the ground truth rubric.

A complete response will:
- Meet ALL success criteria — none can be missing
- Correctly choose whether to ignore, notify, or respond to the email
"""

COMPLETENESS_HUMAN = """
Please grade the following example:

<input>
{input}
</input>

<output>
{output}
</output>

<success_criteria>
{reference}
</success_criteria>
"""

In [ ]:
from utils.utils import format_email_markdown

def completeness_eval(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    email = inputs["email_input"]
    formatted_input = format_email_markdown(
        email["subject"], email["author"], email["to"], email["email_thread"]
    )

    classification = outputs.get("classification_decision", "unknown")
    written_emails = outputs.get("written_emails", [])
    tool_calls = outputs.get("tool_calls", [])

    if written_emails:
        output_lines = [f"Classification: {classification}", "\nEmails written:"]
        for e in written_emails:
            output_lines.append(
                f"  To: {e.get('to')}\n  Subject: {e.get('subject')}\n  Content: {e.get('content', '')}"
            )
    else:
        output_lines = [
            f"Classification: {classification}",
            f"Tools called: {tool_calls}"
        ]
    output_text = "\n".join(output_lines)

    result = judge.invoke([
        {"role": "system", "content": COMPLETENESS_SYSTEM},
        {"role": "human", "content": COMPLETENESS_HUMAN.format(
            input=formatted_input,
            output=output_text,
            reference=reference_outputs.get("response_criteria", "")
        )}
    ])

    return {"key": "completeness", "score": int(result.completeness)}

In [ ]:
response_results = evaluate(
    run_email_agent,
    data=response_dataset,
    evaluators=[completeness_eval],
    experiment_prefix="email-response-v1",
)

response_results.to_pandas()[["outputs.classification_decision", "feedback.completeness"]].head(10)

---
## The Eval Loop: Iterating on Your Agent

Running evals once isn't the goal — the value comes from **iterating**.

```
Run evals → Inspect failures in LangSmith → Hypothesize a fix → Apply fix → Re-run → Compare
```

Common levers to pull:
- **Prompt changes** — add/modify rules, clarify instructions
- **Model changes** — swap to a more capable model
- **Architecture changes** — add a step, change routing logic

### Example: Improving the Triage Prompt

When iterating on a specific step, you can evaluate that step in isolation — no need to re-run the full agent.

Let's say we look at failing triage cases and notice the agent is misclassifying subscription renewal emails as `ignore` instead of `notify`. We can test a prompt fix on just the triage LLM call.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from utils.prompts import get_triage_instructions

class RouterSchema(BaseModel):
    reasoning: str = Field(description="Step-by-step reasoning behind the classification.")
    classification: Literal["ignore", "notify", "respond"] = Field(
        description="The classification of an email."
    )

def make_triage_runner(instructions: str):
    """Returns a run function that applies the given triage prompt.

    Evaluating a single step in isolation is faster and cheaper than
    re-running the full agent — useful for rapid prompt iteration.
    """
    llm_router = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(RouterSchema)

    def run(inputs: dict) -> dict:
        email = inputs["email_input"]
        user_msg = (
            f"From: {email['author']}\n"
            f"To: {email['to']}\n"
            f"Subject: {email['subject']}\n\n"
            f"{email['email_thread']}"
        )
        result = llm_router.invoke([
            {"role": "system", "content": instructions},
            {"role": "user", "content": f"Please classify this email:\n{user_msg}"},
        ])
        return {"classification_decision": result.classification}

    return run

In [ ]:
# v1: baseline prompt
run_triage_v1 = make_triage_runner(get_triage_instructions())

# v2: add an explicit rule for subscription/billing emails
updated_instructions = get_triage_instructions() + """

< Additional Rules >
- Subscription renewal notices and billing notifications should be classified as 'notify' —
  you should be aware of upcoming charges even if no response is needed.
</ Additional Rules >
"""
run_triage_v2 = make_triage_runner(updated_instructions)

# Run both — LangSmith tracks them as separate experiments you can compare side-by-side
results_v1 = evaluate(
    run_triage_v1,
    data=triage_dataset,
    evaluators=[correct_label],
    experiment_prefix="email-triage-prompt-v1",
)

results_v2 = evaluate(
    run_triage_v2,
    data=triage_dataset,
    evaluators=[correct_label],
    experiment_prefix="email-triage-prompt-v2",
)

---
## Summary

We've built a full eval suite for our email agent:

| Eval | Type | What it measures |
|---|---|---|
| `correct_label` | Code (exact match) | Triage classification accuracy |
| `trajectory_match` | Code (set match) | Whether the right tools were called |
| `completeness` | LLM-as-judge | Whether the response meets the success criteria |

### The EDD Flywheel

```
Define success criteria → Build dataset → Run agent → Evaluate → Inspect failures → Improve → Repeat
```

Each iteration, you have **evidence** that things got better (or didn't) — not just intuition.